<a href="https://colab.research.google.com/github/ldebell4/IT371-Smart-Object-Detection/blob/main/preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IT 371 — Smart Traffic Monitoring System
## Data Collection & Preprocessing


---
## 1. Install Libraries
Takes 1-2 minutes. Run every colab session.

In [1]:
!pip install fiftyone tensorflow opencv-python-headless

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.8/112.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 100.5 MB/s eta 0:0

---
## 2. Mount Google Drive
Connects to Google Drive so the processed dataset can be saved.
Allow permissions.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## 3. Imports
Load all the Python libraries needed.

In [9]:
import os
import json
import shutil
import numpy as np
import cv2
from pathlib import Path
import random

print("Imports successful.")

Imports successful.


---
## 4. Configuration
Dataset Settings

If Colab runs out of memory, reduce `IMAGES_PER_CLASS` to 500.

In [19]:
IMAGES_PER_CLASS = 41        # 246 total before augmentation
IMAGE_SIZE       = (224, 224)  # Standard CNN input size
TRAIN_RATIO      = 0.70        # 70% of images go to training
VAL_RATIO        = 0.15        # 15% go to validation
TEST_RATIO       = 0.15        # 15% go to testing
RANDOM_SEED      = 42          # ensure data is the same no matter who runs it

# Where the final dataset will be saved on Google Drive
OUTPUT_DIR = '/content/drive/MyDrive/IT371_Dataset'

# The 6 classes
CLASSES = ['car', 'truck', 'bus', 'motorcycle', 'bicycle', 'person']

print("Configuration loaded.")
print(f"  Images per class : {IMAGES_PER_CLASS}")
print(f"  Total images     : {IMAGES_PER_CLASS * len(CLASSES)}")
print(f"  Image size       : {IMAGE_SIZE}")
print(f"  Split            : {TRAIN_RATIO} train / {VAL_RATIO} val / {TEST_RATIO} test")
print(f"  Output directory : {OUTPUT_DIR}")

Configuration loaded.
  Images per class : 41
  Total images     : 246
  Image size       : (224, 224)
  Split            : 0.7 train / 0.15 val / 0.15 test
  Output directory : /content/drive/MyDrive/IT371_Dataset


---
## 5. Download COCO Dataset
fiftyone connects to COCO's servers and downloads only the images containing our 6 classes meaning we don't have to visit a website or download anything manually.


**This will take 5-15 minutes. Do not close the tab.**

In [14]:
#clear cache
import fiftyone as fo
fo.delete_dataset("coco-2017-train-18000")
print("Cache cleared.")

Cache cleared.


In [20]:
import fiftyone as fo
import fiftyone.zoo as foz

print("Downloading COCO 2017 validation split...")

dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="validation",           # Back to validation — smaller and more reliable
    label_types=["detections"],
    classes=CLASSES,
    max_samples=3000,             # Smaller number — less likely to time out
    seed=RANDOM_SEED,
    shuffle=True,
)

print(f"Download complete. Total samples loaded: {len(dataset)}")

INFO:fiftyone.zoo.datasets:Downloading split 'validation' to '/root/fiftyone/coco-2017/validation' if necessary


Found annotations at '/root/fiftyone/coco-2017/raw/instances_val2017.json'


INFO:fiftyone.utils.coco:Found annotations at '/root/fiftyone/coco-2017/raw/instances_val2017.json'


Only found 2968 (<3000) samples matching your requirements


Sufficient images already downloaded


...

Existing download of split 'validation' is sufficient


INFO:fiftyone.zoo.datasets:Existing download of split 'validation' is sufficient


Loading existing dataset 'coco-2017-validation-3000'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


INFO:fiftyone.zoo.datasets:Loading existing dataset 'coco-2017-validation-3000'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


Download complete. Total samples loaded: 2968


---
## 6. Inspect the Data
Quick check to confirm the images downloaded correctly and labels look right.

In [21]:
print("Sample inspection — first 3 entries:\n")
for i, sample in enumerate(dataset.take(3)):
    labels = []
    if sample.ground_truth is not None:
        labels = list(set([det.label for det in sample.ground_truth.detections]))
    print(f"  Sample {i+1}: {sample.filepath}")
    print(f"    Labels present: {labels}\n")

Sample inspection — first 3 entries:

  Sample 1: /root/fiftyone/coco-2017/validation/data/000000407868.jpg
    Labels present: ['bench', 'person', 'kite', 'frisbee']

  Sample 2: /root/fiftyone/coco-2017/validation/data/000000506454.jpg
    Labels present: ['bench', 'person']

  Sample 3: /root/fiftyone/coco-2017/validation/data/000000329455.jpg
    Labels present: ['chair', 'cup', 'person', 'dining table', 'pizza']



---
## 7. Sort Images by Dominant Class
COCO images can contain multiple object types. We assign each image to whichever
of our 6 classes appears most frequently in that image.

This gives us clean single-label data that our CNN can train on.

In [22]:
def get_dominant_class(sample, classes):
    """Return whichever of our 6 classes appears most in this image."""
    if sample.ground_truth is None:
        return None
    counts = {}
    for det in sample.ground_truth.detections:
        if det.label in classes:
            counts[det.label] = counts.get(det.label, 0) + 1
    if not counts:
        return None
    return max(counts, key=counts.get)

print("Collecting all potential images by dominant class...")

# First, collect all potential images where each class is dominant, without a limit
potential_images_by_dominant_class = {cls: [] for cls in CLASSES}

for sample in dataset:
    dominant = get_dominant_class(sample, CLASSES)
    if dominant is None:
        continue
    potential_images_by_dominant_class[dominant].append(sample.filepath)

print("Selecting up to IMAGES_PER_CLASS for each category...")

class_images = {cls: [] for cls in CLASSES}
for cls in CLASSES:
    random.shuffle(potential_images_by_dominant_class[cls]) # Shuffle before taking top N
    class_images[cls] = potential_images_by_dominant_class[cls][:IMAGES_PER_CLASS]

print("\nImages collected per class:")
for cls, paths in class_images.items():
    print(f"  {cls:12s}: {len(paths)} images")

Selecting up to IMAGES_PER_CLASS for each category...

Images collected per class:
  car         : 41 images
  truck       : 41 images
  bus         : 41 images
  motorcycle  : 41 images
  bicycle     : 41 images
  person      : 41 images


---
## 8. Preprocess and Save Images
For each image this cell will:
1. Read it
2. Resize to 224x224
3. Normalize pixel values from 0-255 to 0.0-1.0
4. Apply augmentation to training images only (flip, brightness)
5. Assign to train, val, or test split
6. Save to your Google Drive

**This is the main processing cell. It will take a few minutes.**

In [23]:
import random
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def augment_image(img):
    """Apply simple augmentation. Returns original + 3 augmented versions."""
    return [
        img,                                      # Original
        cv2.flip(img, 1),                         # Horizontal flip
        np.clip(img * 1.2, 0.0, 1.0),            # Brighter
        np.clip(img * 0.8, 0.0, 1.0),            # Darker
    ]

def preprocess_image(filepath, target_size):
    """Read, resize, and normalize a single image. Returns None if corrupted."""
    try:
        img = cv2.imread(filepath)
        if img is None:
            return None
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, target_size)
        img = img.astype(np.float32) / 255.0
        return img
    except Exception:
        return None

# Create folder structure on Google Drive
splits = ['train', 'val', 'test']
for split in splits:
    for cls in CLASSES:
        Path(f"{OUTPUT_DIR}/{split}/{cls}").mkdir(parents=True, exist_ok=True)

# Stats tracking
stats = {'train': {c: 0 for c in CLASSES}, 'val': {c: 0 for c in CLASSES}, 'test': {c: 0 for c in CLASSES}, 'corrupted': 0}

print("Preprocessing and saving images...")
print("(This will take a few minutes)\n")

for cls, paths in class_images.items():
    random.shuffle(paths)
    n         = len(paths)
    train_end = int(n * TRAIN_RATIO)
    val_end   = train_end + int(n * VAL_RATIO)

    split_map = (
        [(p, 'train') for p in paths[:train_end]] +
        [(p, 'val')   for p in paths[train_end:val_end]] +
        [(p, 'test')  for p in paths[val_end:]]
    )

    for filepath, split in split_map:
        img = preprocess_image(filepath, IMAGE_SIZE)
        if img is None:
            stats['corrupted'] += 1
            continue
        versions = augment_image(img) if split == 'train' else [img]
        for i, version in enumerate(versions):
            base_name = Path(filepath).stem
            out_name  = f"{base_name}_aug{i}.npy" if i > 0 else f"{base_name}.npy"
            np.save(f"{OUTPUT_DIR}/{split}/{cls}/{out_name}", version)
            stats[split][cls] += 1

    print(f"  {cls:12s} done.")

print("\nPreprocessing complete!")

Preprocessing and saving images...
(This will take a few minutes)

  car          done.
  truck        done.
  bus          done.
  motorcycle   done.
  bicycle      done.
  person       done.

Preprocessing complete!


---
## 9. Print Dataset Summary
**Screenshot this output and paste the numbers into your Update 2 slide.**

In [25]:
total_train = sum(stats['train'].values())
total_val   = sum(stats['val'].values())
total_test  = sum(stats['test'].values())
total_all   = total_train + total_val + total_test

print("=" * 58)
print("  DATASET SUMMARY")
print("=" * 58)
print(f"\n  Corrupted images skipped : {stats['corrupted']}")
print(f"\n  {'Class':<14} {'Train':>7} {'Val':>7} {'Test':>7} {'Total':>7}")
print(f"  {'-' * 46}")
for cls in CLASSES:
    t  = stats['train'][cls]
    v  = stats['val'][cls]
    te = stats['test'][cls]
    print(f"  {cls:<14} {t:>7} {v:>7} {te:>7} {t+v+te:>7}")
print(f"  {'-' * 46}")
print(f"  {'TOTAL':<14} {total_train:>7} {total_val:>7} {total_test:>7} {total_all:>7}")
print(f"\n  Split    : {TRAIN_RATIO*100:.0f}% train / {VAL_RATIO*100:.0f}% val / {TEST_RATIO*100:.0f}% test")
print(f"  Size     : {IMAGE_SIZE[0]}x{IMAGE_SIZE[1]} pixels, normalized to [0.0, 1.0]")
print(f"  Augment  : horizontal flip, brightness +20%, brightness -20%")
print(f"  Saved to : {OUTPUT_DIR}")
print("=" * 58)

  DATASET SUMMARY

  Corrupted images skipped : 0

  Class            Train     Val    Test   Total
  ----------------------------------------------
  car                112       6       7     125
  truck              112       6       7     125
  bus                112       6       7     125
  motorcycle         112       6       7     125
  bicycle            112       6       7     125
  person             112       6       7     125
  ----------------------------------------------
  TOTAL              672      36      42     750

  Split    : 70% train / 15% val / 15% test
  Size     : 224x224 pixels, normalized to [0.0, 1.0]
  Augment  : horizontal flip, brightness +20%, brightness -20%
  Saved to : /content/drive/MyDrive/IT371_Dataset


---
## 10. Save Metadata
This saves a small JSON file to your Google Drive that tells Adam exactly
what the dataset contains so he can load it without re-running everything.

After this cell finishes, **share your IT371_Dataset folder on Google Drive with Adam.**

In [26]:
metadata = {
    "classes"      : CLASSES,
    "image_size"   : list(IMAGE_SIZE),
    "split_ratios" : {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
    "random_seed"  : RANDOM_SEED,
    "counts"       : {"train": stats['train'], "val": stats['val'], "test": stats['test']},
    "output_dir"   : OUTPUT_DIR,
    "augmentation" : ["horizontal_flip", "brightness_+20pct", "brightness_-20pct"],
    "normalization": "pixel values divided by 255.0, range [0.0, 1.0]",
}

with open(f"{OUTPUT_DIR}/dataset_metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved.")
print(f"\nNext step: Share this folder with Adam on Google Drive:")
print(f"  {OUTPUT_DIR}")
print("\nYou are done! Preprocessing complete.")

Metadata saved.

Next step: Share this folder with Adam on Google Drive:
  /content/drive/MyDrive/IT371_Dataset

You are done! Preprocessing complete.
